# LLM Agent for answering questions based on my research papers

In [1]:
# Step 1 - Load all the papers as LangChain documents using the pypdfloader

import glob
from langchain_community.document_loaders import PyPDFLoader

file_paths = glob.glob('files/*.pdf')
all_docs = []

def clean_doc(doc):
    # Encode to bytes ignoring errors, then decode back to str
    doc.page_content = doc.page_content.encode('utf-8', 'ignore').decode('utf-8', 'ignore')
    return doc

for file_path in file_paths:
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    clean_docs = []
    for doc in docs:
        clean_docs.append(clean_doc(doc)) 
    all_docs.extend(clean_docs)

print(len(all_docs))

70


## Sanity check

In [2]:
print(f"{all_docs[0].page_content[:200]}\n")
print(all_docs[0].metadata)

Evaluating the Impact of Personalized Value Alignment in 
Human-Robot Interaction: Insights into Trust and Team 
Performance Outcomes 
Shreyas Bhat Joseph B. Lyons 
shreyasb@umich.edu joseph.lyons.6@u

{'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5; modified using iText 4.2.0 by 1T3XT', 'creator': 'LaTeX with acmart 2023/03/30 v1.90 Typesetting articles for the Association for Computing Machinery and hyperref 2023-07-08 v7.01b Hypertext links for LaTeX', 'creationdate': '2023-12-15T16:51:17+00:00', 'keywords': 'Human-robot teaming, trust-aware decision-making, value-alignment', 'moddate': '2025-11-17T07:23:44-08:00', 'trapped': '/False', 'subject': '-  Human-centered computing  ->  Empirical studies in HCI.-  Computer systems organization  ->  Robotic autonomy.', 'pdfversion': '1.5', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'Evaluating the Impact of Personalized Va

# Text splitting 
For both information retrieval and downstream question-answering purposes, a page may be too coarse a representation. Our goal in the end will be to retrieve Document objects that answer an input query, and further splitting our PDF will help ensure that the meanings of relevant portions of the document are not “washed out” by surrounding text.

We can use text splitters for this purpose. Here we will use a simple text splitter that partitions based on characters. We will split our documents into chunks of 1000 characters with 200 characters of overlap between chunks. The overlap helps mitigate the possibility of separating a statement from important context related to it. We use the RecursiveCharacterTextSplitter, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases. 

We set add_start_index=True so that the character index where each split Document starts within the initial Document is preserved as metadata attribute “start_index”.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(all_docs)

print(all_splits[1])

page_content='presumes the human’s reward function mirrors its own; a non-
adaptive-learner strategy in which the robot learns the human’s 
reward function for trust estimation and human behavior modeling, 
but still optimizes its own reward function; and an adaptive-learner 
strategy in which the robot learns the human’s reward function 
and adopts it as its own. Two human-subject experiments with a 
total number of  = 54 participants were conducted. In both ex-
periments, the human-robot team searches for potential threats in 
a town. The team sequentially goes through search sites to look for 
threats. We model the interaction between the human and the robot 
as a trust-aware Markov Decision Process (trust-aware MDP) and 
use Bayesian Inverse Reinforcement Learning (IRL) to estimate the 
reward weights of the human as they interact with the robot. In Ex-
periment 1, we start our learning algorithm with an informed prior 
of the human’s values/goals. In Experiment 2, we start the lea

# Get embeddings from the text data

Vector search is a common way to store and search over unstructured data (such as unstructured text). The idea is to store numeric vectors that are associated with the text. Given a query, we can embed it as a vector of the same dimension and use vector similarity metrics (such as cosine similarity) to identify related text.

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [5]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 3072

[-0.0028756672982126474, 0.0010072204750031233, 0.01650209166109562, -0.057829152792692184, -0.0025247696321457624, -0.0031135950703173876, -0.0007002169732004404, 0.020272426307201385, 0.006575762294232845, -0.012024512514472008]


# Vector stores
LangChain VectorStore objects contain methods for adding text and Document objects to the store, and querying them using various similarity metrics. They are often initialized with embedding models, which determine how text data is translated to numeric vectors. 

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

# Instantiate the vector store
vector_store = InMemoryVectorStore(embeddings)

In [7]:
# Store the documents in the vector store
ids = vector_store.add_documents(documents=all_splits)

In [8]:
# Return documents based on similarity to a string query:

results = vector_store.similarity_search(
    "When does the adaptive-learner strategy result in high trust?"
)

print(results[0])

page_content='non-adaptive learner strategy ( .p = 0.003 and .p< 0.001, respectively).
Regarding the number of agreements (Fig. 7c), there was a signiﬁcant difference
among the three strategies .(F(1.584,36.435) = 25.829,p< 0.001). Post hoc
analysis showed that there was a signiﬁcant difference between the non-learner and
the adaptive-learner strategies .(p < 0.001) and between the non-adaptive-learner
and adaptive-learner strategies .(p < 0.001).
Comparing reliance intentions (Fig. 7d), there was a signiﬁcant difference
between the three strategies ( .F(2,46) = 13.691,p < 0.001), with the adaptive-
learner strategy rated higher than the non-learner strategy ( .p< 0.001) and the
non-adaptive-learner strategy ( .p = 0.004).' metadata={'producer': 'Acrobat DC', 'creator': 'LaTeX with hyperref package + hypdvips', 'creationdate': '2024-10-12T14:32:59+05:30', 'author': '', 'keywords': '', 'moddate': '2024-10-23T13:21:52+05:30', 'subject': '', 'title': 'Value Alignment and Trust in Human-Ro

In [10]:
# Async query
results = await vector_store.asimilarity_search("What were the three interaction strategies used in 'Effects of Personalized Value Alignment' paper?")
print(results[0])

page_content='4.3 Interaction Strategies
We designed three interaction strategies for the intelligent
agent:
• Non-learner: The intelligent agent does not learn the re-
ward weights of the human. It assumes that the human
and the intelligent agent share the same reward weights.
• Non-adaptive learner:The intelligent agent learns per-
sonalized reward weights for each human. It only uses
these learned weights for performance assessment and
human behavior modeling. It still optimizes the MDP
based on its own fixed reward weights.
• Adaptive learner: The intelligent agent learns person-
alized reward weights for each human. It uses them for
performance assessment, human behavior modeling, and
also optimizes the MDP based on these reward weights.
In other words, it updates its own reward function match
the learned reward function.
Although it may look like the non-learner and the non-
adaptive learner both optimize the same reward function,
they actually optimize expected reward under the 

In [11]:
# Return scores about the similarity. It is a distance metric that varies inversely with similarity

results = vector_store.similarity_search_with_score("When was the study on 'Effects of Learning State Dependence of Reward Weights' published?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)


Score: 0.7359769733133782

page_content='Effects of Learning State Dependence of Reward
W eights on Trust and T eam Performance in a
Human-Robot Sequential Decision-Making T ask
Shreyas Bhat ∗, Joseph B. Lyons † , Cong Shi ‡ and X. Jessie Y ang ∗
∗Industrial and Operations Engineering, University of Michigan, Ann Arbor, MI
† Air Force Research Laboratory , Dayton, OH
‡ Herbert Business School, University of Miami, Miami, FL
Abstract—In this paper , we evaluate two interaction strategies
for a robot in a sequential decision-making task: one which uses
a state-dependent reward function and the other that uses a
state-independent (constant) reward function. T owards this, we
present a study done on Amazon Mechanical T urk to learn the
state-dependent reward function. Using this reward function,
we compare the two strategies in simulation, where we also set
the risk levels actively to induce a difference between the two
strategies. Our results indicate that the interaction strategy using' 

In [12]:
# return documents based on similarity to an embedded query

embedding = embeddings.embed_query("What was the title of the study publised in 2022?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='IEEE ROBOTICS AND AUTOMATION LETTERS, VOL. 7, NO. 4, OCTOBER 2022 8815
Clustering Trust Dynamics in a Human-Robot
Sequential Decision-Making Task
Shreyas Bhat , Graduate Student Member , IEEE, Joseph B. Lyons, Cong Shi, and X. Jessie Yang
Abstract—In this paper, we present a framework for trust-aware
sequential decision-making in a human-robot team wherein the
human agent’s trust in the robotic agent is dependent on the reward
obtained by the team. We model the problem as a ﬁnite-horizon
Markov Decision Process with the trust of the human on the robot
as a state variable. We develop a reward-based performance metric
to drive the trust update model, allowing the robotic agent to make
trust-aware recommendations. We conduct a human-subject ex-
periment with a total of 45 participants and analyze how the human
agent’s trust evolves over time. Results show that the proposed trust
update model is able to accurately capture the human agent’s trust' metadata={'producer': 'Acroba

# Retrievers
LangChain VectorStore objects do not subclass @[Runnable]. LangChain @[Retrievers] are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations). Although we can construct retrievers from vector stores, retrievers can interface with non-vector store sources of data, as well (such as external APIs).

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:


In [13]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "What are the three types of trust dynamics found?",
        "What is the publication year of my most recent paper?",
    ],
)


/home/shreyas-bhat/anaconda3/envs/py_llms/lib/python3.13/site-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


[[Document(id='defb90c7-6849-4f5d-aca8-6be07122c68b', metadata={'producer': 'Acrobat Distiller 11.0 (Windows); modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2022-07-15T00:09:50+05:30', 'moddate': '2022-07-16T16:27:18-04:00', 'ieee article id': '9816108', 'ieee issue id': '9831196', 'subject': 'IEEE Robotics and Automation Letters;2022;7;4;10.1109/LRA.2022.3188902', 'ieee publication id': '7083369', 'title': 'Clustering Trust Dynamics in a Human-Robot Sequential Decision-Making Task', 'source': 'files/paper1.pdf', 'total_pages': 8, 'page': 6, 'page_label': '7', 'start_index': 1591}, page_content='trends, unfortunately, did not reach signiﬁcance.\nPerforming one way ANOV A on the post-experiment mea-\nsures show signiﬁcant difference between the three types of\ntrust dynamics in their post-experiment trust reports (Trust\nquestionnaire by Muir and Moray F(2,42) = 22.167,p<\n0.001, Trust questionnaire by Lyons and GuznovF(2,42)